# 02. Feature Engineering
Обоснование выбора 8 структурных признаков.

In [ ]:
import sys; sys.path.insert(0, "..")
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from feature_extractor import PolynomialSystemFeatureExtractor, BENCHMARK_SYSTEMS
from rl_monomial_agent import RLEnvironment

## 1. Описание признаков

In [ ]:
ext = PolynomialSystemFeatureExtractor()
descriptions = {
    "n_vars_norm":        "Нормализованное число переменных",
    "max_degree_norm":    "Нормализованная максимальная степень",
    "avg_degree_norm":    "Нормализованная средняя степень",
    "n_polys_norm":       "Нормализованное число полиномов",
    "avg_monomials_norm": "Нормализованное среднее число мономов",
    "density":            "Плотность системы",
    "sparsity":           "1 - density",
    "complexity_index":   "Индекс сложности (оценка S-пар)",
}
for name, desc in descriptions.items():
    print(f"{name:25s}: {desc}")

## 2. Корреляция признаков с временем GREVLEX

In [ ]:
feats = np.stack([ext.extract(system_dict=v) for v in BENCHMARK_SYSTEMS.values()])
bench = RLEnvironment.BENCHMARK_TIMES
times_g = np.array([bench[k]["grevlex"] for k in BENCHMARK_SYSTEMS])
corrs = [np.corrcoef(feats[:, i], times_g)[0, 1] for i in range(8)]
corr_df = pd.DataFrame({"feature": ext.feature_names(), "correlation": corrs})
corr_df = corr_df.reindex(corr_df["correlation"].abs().sort_values(ascending=False).index)
colors = ["#e74c3c" if c > 0 else "#3498db" for c in corr_df["correlation"]]
plt.figure(figsize=(8, 4))
plt.barh(corr_df["feature"], corr_df["correlation"], color=colors)
plt.axvline(0, color="black", lw=0.8)
plt.title("Корреляция признаков с временем GREVLEX")
plt.tight_layout(); plt.show()
print(corr_df.to_string(index=False))

## 3. Матрица признаков всех систем

In [ ]:
feat_df = pd.DataFrame(feats, columns=ext.feature_names(),
                       index=list(BENCHMARK_SYSTEMS.keys()))
print(feat_df.round(3).to_string())